# Smart Housing Inspection Dashboard V1 — Development Controller

**Purpose:** This notebook is the development controller for the V1 project.

## Source-of-truth policy

- 🟢 **GitHub repository is the persistent source of truth.**
- 🟢 Project `.py` files are the actual application source code.
- 🟢 Colab is a temporary development, testing, and runtime environment.
- 🔴 This notebook does **not** contain duplicate canonical application source code.
- 🔴 This notebook does **not** recreate project source files from a recovery bundle.
- 🟡 The notebook may run tests, start the application, inspect Git state, and document the development session.

## Development workflow

**PLAN → DESIGN → CODE → RUN → TEST → FIX → VERIFY → APPROVE → COMMIT → PUSH → NEXT**

## Safety rules

1. Never silently overwrite local changes.
2. Never reset or delete inspection data during normal verification.
3. Never store GitHub tokens/API keys in notebook source, Git remotes, or project files.
4. Commit only after the relevant tests and runtime checks pass.
5. Keep `main` stable and verified.
6. Make small, logical commits.


## 1. Project Configuration

🟢 Permanent controller configuration.

In [1]:
# ============================================================
# 1. Project Configuration
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import time
import json

PROJECT_NAME = "Smart_Housing_Inspection_V1"
PROJECT_ROOT = Path("/content") / PROJECT_NAME
GITHUB_REPO = "https://github.com/saadsaeed1/Smart_Housing_Inspection_V1.git"
GITHUB_BRANCH = "main"
STREAMLIT_PORT = 8501

print(f"Project: {PROJECT_NAME}")
print(f"Project root: {PROJECT_ROOT}")
print(f"GitHub branch: {GITHUB_BRANCH}")
print("🟢 Controller configuration loaded")


Project: Smart_Housing_Inspection_V1
Project root: /content/Smart_Housing_Inspection_V1
GitHub branch: main
🟢 Controller configuration loaded


## 2. GitHub Bootstrap / Synchronization

🟢 **Permanent daily workflow**

This cell makes GitHub the bootstrap source.

- Fresh Colab: clone the repository.
- Existing project: inspect Git state first.
- Clean working tree: fast-forward pull.
- Local changes: stop instead of overwriting them.

The authentication helper uses the optional Colab Secret `GH_TOKEN` without writing the token into the repository or remote URL.


In [3]:
# ============================================================
# 2. GitHub Bootstrap / Synchronization
# ============================================================

import subprocess
import os
from pathlib import Path

def run_cmd(cmd, cwd=None, env=None, check=True):
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result

def github_env():
    env = os.environ.copy()
    # Optional secure authentication through Colab Secret GH_TOKEN.
    # Token is passed through an in-memory HTTP header and is never saved to Git config.
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception:
        token = None

    if token:
        env["GIT_HTTP_EXTRAHEADER"] = f"Authorization: Bearer {token}"
        print("🟢 GitHub authentication available through Colab Secret")
    else:
        print("🟡 GH_TOKEN not available — attempting normal GitHub access")
    return env

env = github_env()

if not PROJECT_ROOT.exists():
    print("Fresh Colab runtime detected — cloning GitHub repository...")
    run_cmd(
        ["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(PROJECT_ROOT)],
        env=env,
    )
    print("🟢 GITHUB CLONE PASSED")
else:
    print("Existing project directory detected.")

    status = run_cmd(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
        env=env,
    )

    if status.stdout.strip():
        print("🔴 LOCAL CHANGES DETECTED")
        print("Do not pull automatically. Review/commit/stash the changes first.")
        raise RuntimeError("Unsafe to synchronize: local Git changes are present.")

    branch = run_cmd(
        ["git", "branch", "--show-current"],
        cwd=PROJECT_ROOT,
        env=env,
    ).stdout.strip()

    if branch != GITHUB_BRANCH:
        raise RuntimeError(
            f"Expected branch '{GITHUB_BRANCH}', but current branch is '{branch}'."
        )

    run_cmd(
        ["git", "pull", "--ff-only", "origin", GITHUB_BRANCH],
        cwd=PROJECT_ROOT,
        env=env,
    )
    print("🟢 GITHUB FAST-FORWARD SYNC PASSED")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"🟢 Working directory: {PROJECT_ROOT}")


🟢 GitHub authentication available through Colab Secret
Existing project directory detected.
main
Already up to date.
From https://github.com/saadsaeed1/Smart_Housing_Inspection_V1
 * branch            main       -> FETCH_HEAD
🟢 GITHUB FAST-FORWARD SYNC PASSED
🟢 Working directory: /content/Smart_Housing_Inspection_V1


## 3. Environment & Dependencies

🟢 Verify the required runtime without recreating project source files.

In [4]:
# ============================================================
# 3. Environment & Dependencies
# ============================================================

requirements_file = PROJECT_ROOT / "requirements.txt"

assert requirements_file.is_file(), "requirements.txt is missing."

print("Required project files are available from GitHub.")
print("\nrequirements.txt:")
print(requirements_file.read_text(encoding="utf-8"))

# Install exact project dependencies when needed.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
    check=True,
)

print("🟢 DEPENDENCY INSTALLATION / VERIFICATION PASSED")


Required project files are available from GitHub.

requirements.txt:
pandas==2.2.3
streamlit==1.63.0
plotly==5.24.1
openpyxl==3.1.5
fpdf2==2.8.8
Pillow==11.3.0

🟢 DEPENDENCY INSTALLATION / VERIFICATION PASSED


## 4. Project Structure Verification

🟡 Non-destructive verification.

In [6]:
# ============================================================
# 4. Project Structure Verification
# ============================================================

required_files = [
    "app.py",
    "config.py",
    "schema.py",
    "data_manager.py",
    "validation.py",
    "requirements.txt",
    "README.md",
    ".gitignore",
    "pages/__init__.py",
    "pages/inspections.py",
]

# These directories are required by the local application runtime.
# Their contents are intentionally gitignored.
runtime_dirs = [
    "data",
    "assets",
    "assets/site_images",
]

required_dirs = [
    "pages",
    "data",
    "assets",
    "assets/site_images",
]

# ------------------------------------------------------------
# Create local runtime directories if they are not present.
# Git does not track empty directories.
# ------------------------------------------------------------

for rel in runtime_dirs:
    path = PROJECT_ROOT / rel
    path.mkdir(parents=True, exist_ok=True)
    print(f"🟢 runtime directory ready: {rel}")

# ------------------------------------------------------------
# Verify required directories
# ------------------------------------------------------------

for rel in required_dirs:
    assert (PROJECT_ROOT / rel).is_dir(), f"Missing directory: {rel}"
    print(f"🟢 directory: {rel}")

# ------------------------------------------------------------
# Verify required source/configuration files
# ------------------------------------------------------------

for rel in required_files:
    assert (PROJECT_ROOT / rel).is_file(), f"Missing file: {rel}"
    print(f"🟢 file: {rel}")

print("🟢 PROJECT STRUCTURE CHECK PASSED")

🟢 runtime directory ready: data
🟢 runtime directory ready: assets
🟢 runtime directory ready: assets/site_images
🟢 directory: pages
🟢 directory: data
🟢 directory: assets
🟢 directory: assets/site_images
🟢 file: app.py
🟢 file: config.py
🟢 file: schema.py
🟢 file: data_manager.py
🟢 file: validation.py
🟢 file: requirements.txt
🟢 file: README.md
🟢 file: .gitignore
🟢 file: pages/__init__.py
🟢 file: pages/inspections.py
🟢 PROJECT STRUCTURE CHECK PASSED


## 5. Python Syntax & Import Verification

🟡 Non-destructive verification.


In [7]:
# ============================================================
# 5. Python Syntax & Import Verification
# ============================================================

import py_compile
import importlib

python_files = [
    PROJECT_ROOT / "app.py",
    PROJECT_ROOT / "config.py",
    PROJECT_ROOT / "schema.py",
    PROJECT_ROOT / "data_manager.py",
    PROJECT_ROOT / "validation.py",
    PROJECT_ROOT / "pages" / "inspections.py",
]

for file in python_files:
    py_compile.compile(str(file), doraise=True)
    print(f"🟢 syntax: {file.relative_to(PROJECT_ROOT)}")

for module_name in ["config", "schema", "data_manager", "validation"]:
    module = importlib.import_module(module_name)
    importlib.reload(module)
    print(f"🟢 import: {module_name}")

print("🟢 PYTHON SYNTAX & IMPORT CHECK PASSED")


🟢 syntax: app.py
🟢 syntax: config.py
🟢 syntax: schema.py
🟢 syntax: data_manager.py
🟢 syntax: validation.py
🟢 syntax: pages/inspections.py
🟢 import: config
🟢 import: schema
🟢 data_manager.py loaded successfully.
🟢 data_manager.py loaded successfully.
🟢 import: data_manager
🟢 validation.py loaded successfully.
🟢 validation.py loaded successfully.
🟢 import: validation
🟢 PYTHON SYNTAX & IMPORT CHECK PASSED


## 6. Automated Test Runner

🟡 **Testing gate**

The project should keep repeatable tests under `tests/`. This controller runs them when they exist.

If the repository does not yet contain automated test files, the controller reports the gap instead of pretending the test suite passed.


In [8]:
# ============================================================
# 6. Automated Test Runner
# ============================================================

tests_dir = PROJECT_ROOT / "tests"
test_files = sorted(tests_dir.glob("test_*.py")) if tests_dir.exists() else []

if not test_files:
    print("🟡 NO AUTOMATED TEST FILES FOUND")
    print("Backend verification can still be performed manually,")
    print("but this milestone is not yet a full pytest test suite.")
else:
    print("Automated tests:")
    for test_file in test_files:
        print(f"  - {test_file.relative_to(PROJECT_ROOT)}")

    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=str(PROJECT_ROOT),
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError("🔴 AUTOMATED TEST SUITE FAILED")

    print("🟢 AUTOMATED TEST SUITE PASSED")


🟡 NO AUTOMATED TEST FILES FOUND
Backend verification can still be performed manually,
but this milestone is not yet a full pytest test suite.


## 7. Backend Smoke Test

🟡 Fast, non-destructive health check.

In [9]:
# ============================================================
# 7. Backend Smoke Test
# ============================================================

from datetime import date
from data_manager import normalize_inspection
from validation import validate_inspection

record = {
    "Inspection ID": "",
    "Inspection Date": date(2026, 9, 8),
    "Inspector": "Test Inspector",
    "Sector": "Sector A",
    "Plot Number": "A-101",
    "Owner": "Test Owner",
    "Contractor": "",
    "Inspection Type": "Routine Inspection",
    "Construction Activity": "Brickwork",
    "Level / Floor": "Ground Floor",
    "Progress %": "75",
    "Observation": "Test observation.",
    "Defects": "",
    "Violation Type": "No Violation",
    "Severity": "",
    "Compliance Status": "Compliant",
    "Recommended Action": "",
    "Deadline": "",
    "Work Stopped": "No",
    "Follow-up Required": False,
    "Follow-up Date": "",
    "Front-view Site Image": "assets/site_images/test.jpg",
}

normalized = normalize_inspection(record)
errors = validate_inspection(normalized)

assert normalized["Inspector"] == "Test Inspector"
assert normalized["Progress %"] == 75.0
assert normalized["Work Stopped"] is False
assert normalized["Inspection Date"] == "2026-09-08"
assert errors == []

print("🟢 NORMALIZATION CHECK PASSED")
print("🟢 VALIDATION CHECK PASSED")
print("🟢 NON-DESTRUCTIVE BACKEND SMOKE TEST PASSED")


🟢 NORMALIZATION CHECK PASSED
🟢 VALIDATION CHECK PASSED
🟢 NON-DESTRUCTIVE BACKEND SMOKE TEST PASSED


## 8. Streamlit Runtime

🟢 Start or reuse the local Streamlit application.

This does not expose the application publicly. Public browser access is handled separately by the optional tunnel section.


In [10]:
# ============================================================
# 8. Streamlit Runtime
# ============================================================

import requests

APP_FILE = PROJECT_ROOT / "app.py"

def streamlit_is_running():
    try:
        return requests.get(
            f"http://127.0.0.1:{STREAMLIT_PORT}",
            timeout=3
        ).status_code == 200
    except Exception:
        return False

if streamlit_is_running():
    print("🟢 Existing Streamlit server detected — reusing it.")
else:
    streamlit_process = subprocess.Popen(
        [
            "streamlit", "run", str(APP_FILE),
            "--server.port", str(STREAMLIT_PORT),
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)

if not streamlit_is_running():
    raise RuntimeError("🔴 Streamlit did not respond on the expected port.")

print(f"🟢 STREAMLIT HTTP CHECK PASSED — port {STREAMLIT_PORT}")


🟢 STREAMLIT HTTP CHECK PASSED — port 8501


## 9. Optional Cloudflare Quick Tunnel

🟡 Browser exposure only.

Run this only when browser testing is required. The tunnel is temporary and is **not** part of production deployment.


In [11]:
# ============================================================
# 9. Optional Cloudflare Quick Tunnel
# ============================================================
# Run this cell manually when browser access from outside Colab is needed.
# It intentionally does not run automatically.

print("🟡 OPTIONAL CELL")
print("Start the Cloudflare Quick Tunnel only when browser testing is required.")
print(f"Target: http://127.0.0.1:{STREAMLIT_PORT}")


🟡 OPTIONAL CELL
Start the Cloudflare Quick Tunnel only when browser testing is required.
Target: http://127.0.0.1:8501


## 10. Git Review

🟡 **Pre-commit gate**

Review changes before committing. The controller never automatically commits or pushes application changes.


In [12]:
# ============================================================
# 10. Git Review
# ============================================================

print("Current branch:")
run_cmd(["git", "branch", "--show-current"], cwd=PROJECT_ROOT)

print("\nGit status:")
run_cmd(["git", "status", "--short"], cwd=PROJECT_ROOT)

print("\nRecent commits:")
run_cmd(["git", "log", "--oneline", "-5"], cwd=PROJECT_ROOT)

print("\nRemote:")
run_cmd(["git", "remote", "-v"], cwd=PROJECT_ROOT)

print("\n🟡 Review the output above before committing.")


Current branch:
main

Git status:

Recent commits:
12eb18e Initial V1 project foundation

Remote:
origin	https://github.com/saadsaeed1/Smart_Housing_Inspection_V1.git (fetch)
origin	https://github.com/saadsaeed1/Smart_Housing_Inspection_V1.git (push)

🟡 Review the output above before committing.


## 11. Commit & Push Workflow

🟢 Manual approval required.

Use small logical commits. Do not put a GitHub token into the remote URL.

Recommended sequence:

```text
git status
git diff
pytest
Streamlit/browser verification
git add <specific files>
git commit -m "Short logical message"
git push origin main
```

The notebook intentionally does **not** auto-commit or auto-push.


## 12. Session Handoff

At the end of each development session, record:

- Current milestone
- What was completed
- Tests passed
- Browser/runtime result
- Files changed
- Commit hash
- Next step
- Known issues / decisions

### Current milestone

**Phase 2 — Development Controller Refactoring**

A milestone is complete only after its verification evidence is available.


In [13]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nNotebooks in project root:")
for p in PROJECT_ROOT.glob("*.ipynb"):
    print("🟢", p.name)

print("\nNotebooks in /content:")
for p in Path("/content").glob("*.ipynb"):
    print("📓", p.name)

Current working directory:
/content/Smart_Housing_Inspection_V1

Project root:
/content/Smart_Housing_Inspection_V1

Notebooks in project root:

Notebooks in /content:


In [ ]:
# ============================================================
# 12. Session Handoff Template
# ============================================================

handoff = {
    "project": PROJECT_NAME,
    "milestone": "Phase 2 — Development Controller Refactoring",
    "current_step": "Step A–G controller refactoring",
    "completed": [],
    "tests_passed": [],
    "runtime_verified": False,
    "files_changed": [],
    "commit": "",
    "next_step": "",
    "known_issues": [],
}

print(json.dumps(handoff, indent=2))


## 13. Development Rules

### 🟢 Permanent rules

- GitHub is the persistent source of truth.
- `.py` files are the application source.
- Colab is temporary.
- Tests are repeatable and non-destructive.
- Small logical commits.
- `main` stays stable.
- No secrets in source code or Git history.

### 🔴 No longer part of normal development

- Embedded recovery bundles
- Notebook canonical source-code cells
- Repeated `git init`
- Repeated remote creation
- Duplicate replacement/fix source cells
- Destructive test resets
- Automatically overwriting local work

### Development sequence

**PLAN → DESIGN → CODE → RUN → TEST → FIX → VERIFY → APPROVE → COMMIT → PUSH → NEXT**
